In [ ]:
#!pip install pyspark findspark

In [1]:
import site
print(site.getsitepackages()[0])


c:\Users\ADMIN\anaconda3


In [16]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.getOrCreate()


In [3]:
df = spark.createDataFrame([(1, 'a'), (2, 'b')], schema=['id', 'name'])
df.show(1)

+---+----+
| id|name|
+---+----+
|  1|   a|
+---+----+
only showing top 1 row


In [12]:
df3 = spark.createDataFrame([(1, 'a',1), (2, 'b',1),(3,'c',3)], schema=['id', 'name','id1'])
df3

DataFrame[id: bigint, name: string, id1: bigint]

In [18]:
import pandas as pd
d={'name':['sanjeev','sisi','ayya','reddy','seelam'],
   'id':[1,2,4,5,6,]}

pdf=pd.DataFrame(d)
#pdf

sdf=spark.createDataFrame(pdf)
sdf

DataFrame[name: string, id: bigint]

# Spark Session

In [ ]:
from pyspark.sql import SparkSession

# Initialize your Spark Session safely
spark = SparkSession.builder \
    .appName("VSCodeNotebookApp") \
    .master("local[*]") \
    .getOrCreate()

print("👉 Spark Session Active! UI available at:", spark.sparkContext.uiWebUrl)


# Sub-section here

In [ ]:
import os
import sys
from pyspark.sql import SparkSession


# 2. Force Python to bind directly to the active kernel path
os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable

# 3. Initialize the session using static, open local loopback ports
spark = SparkSession.builder \
    .master("local[1]") \
    .appName("NoAdminFirewallFix1") \
    .config("spark.driver.host", "127.0.0.1") \
    .config("spark.driver.bindAddress", "127.0.0.1") \
    .config("spark.driver.port", "7077") \
    .config("spark.blockManager.port", "6060") \
    .config("spark.driver.extraJavaOptions", 
            "--add-opens=java.base/java.nio=ALL-UNNAMED "
            "--add-opens=java.base/sun.nio.ch=ALL-UNNAMED "
            "--add-opens=java.base/java.lang=ALL-UNNAMED") \
    .getOrCreate()

# 4. Run the DataFrame action
df = spark.createDataFrame([(1, 'a'), (2, 'b')], schema=['id', 'name'])
df.show(1)


In [ ]:
import os
import sys
from pyspark.sql import SparkSession

# 1. Clear any stuck driver states
try:
    spark.stop()
except:
    pass

# 2. Force Python to bind directly to the active kernel path
os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable

# 3. Initialize the session using static, open local loopback ports
spark = SparkSession.builder \
    .master("local[1]") \
    .appName("NoAdminFirewallFix") \
    .config("spark.driver.host", "127.0.0.1") \
    .config("spark.driver.bindAddress", "127.0.0.1") \
    .config("spark.driver.port", "7077") \
    .config("spark.blockManager.port", "6060") \
    .config("spark.driver.extraJavaOptions", 
            "--add-opens=java.base/java.nio=ALL-UNNAMED "
            "--add-opens=java.base/sun.nio.ch=ALL-UNNAMED "
            "--add-opens=java.base/java.lang=ALL-UNNAMED") \
    .getOrCreate()

# 4. Run the DataFrame action
df = spark.createDataFrame([(1, 'a'), (2, 'b')], schema=['id', 'name'])
df.show()


In [ ]:
import pyspark
import os

# 1. Print the exact hidden folder location
pyspark_path = os.path.dirname(pyspark.__file__)
print("Your hidden Spark path is:")
print(pyspark_path)


# 📓 Troubleshooting Log: Local Windows PySpark Setup (Java 17 & Pip)

This log summarizes the environment bugs faced, diagnostic commands used, and the permanent background resolution applied to this workspace. 

---

### 🚨 1. Module Not Found / Command Not Recognized
* **The Problem:** Running `python` or `pyspark` returned a *"not recognized as an internal or external command"* terminal error.
* **The Cause:** Windows did not know where the installation folders lived because they were missing from the system environment paths.
* **Diagnostic Commands:**
  ```cmd
  # Run in Command Prompt (CMD) to check active system paths
  echo %PATH%
  ```
  ```powershell
  # Run in PowerShell to see a clean, split list of paths
  $env:Path -split ';'
  ```

### 🚨 1. VS Code: PySpark Module Not Found
* **The Problem:** Running code in your VS Code Jupyter Notebook returned a *"ModuleNotFoundError: No module named 'pyspark'"*.
* **The Cause:** The package was missing from the environment your notebook kernel was using, or the workspace lacked a bridge tool to locate the installations.
* **Diagnostic & Fix Command (Run in Notebook Cell):**
  ```python
  !pip install pyspark findspark
  ```
* **The Resolution:** Ran the installation command directly inside the notebook cell to force-install both `pyspark` and the `findspark` helper tool into your active runtime environment.


* **The Resolution:** Added the main Python installation folder and its `\Scripts` subfolder to the **Windows User Environment Variables**, then restarted the IDE.

---

### 📂 2. Finding Hidden Pip Installation Paths
* **The Problem:** Attempting to create configuration files inside a traditional Spark folder layout failed because PySpark was installed via `pip`, which hides its package files.
* **Diagnostic Command (Run in Notebook Cell):**
  ```python
  import pyspark
  import os
  print("Hidden PySpark Package Location:")
  print(os.path.dirname(pyspark.__file__))
  ```
* **The Resolution:** Ran the script to find the hidden path (deep within `site-packages\pyspark`), confirming we had to handle configurations through Python rather than external Windows files.

---

### ☕ 3. Java 17 Module Access Restrictions
* **The Problem:** The Spark session initialized successfully, but the moment data processing occurred (e.g., `df.show()`), the cell hung indefinitely.
* **The Cause:** Java 16+ enforces strict modular encapsulation. It blocks external languages (like Python) from accessing its core memory blocks unless explicitly allowed.
* **Diagnostic Command (Run in Notebook Cell):**
  ```python
  !java -version
  ```
* **The Resolution:** Passed strict memory-opening override flags (`--add-opens=java.base/java.nio=ALL-UNNAMED`) directly into the JVM arguments to allow safe data streaming.

---

### 🌐 4. Network LAN Loop Mismatch (`.lan` address)
* **The Problem:** The Spark master engine bound itself to your Local Area Network web address: `http://desktop-e3jqhst.lan`. Python failed to communicate back, throwing a `SocketTimeoutException`.
* **The Cause:** Spark automatically broadcasts to your router's LAN domain, but Python's worker script attempts to establish communication via `localhost` (`127.0.0.1`). 
* **Diagnostic Command (Run in CMD/PowerShell):**
  ```cmd
  ping localhost
  ```
* **The Resolution:** Hardcoded the network binding addresses directly to `127.0.0.1` inside the system config parameters to force an offline loopback.

---

### 🛡️ 5. Firewall "Access Denied" Block
* **The Problem:** Windows Firewall blocked the random network ports spawned by Spark workers to process data rows. 
* **The Cause:** Modifying the Windows Firewall requires local Administrator clearance, which your profile lacks.
* **Diagnostic Command (Run in PowerShell - Returned 'Access Denied'):**
  ```powershell
  New-NetFirewallRule -DisplayName "Allow PySpark" -Direction Inbound -Action Allow -Protocol TCP -LocalAddress 127.0.0.1
  ```
* **The Resolution:** Instead of touching the firewall, we modified Spark's execution architecture. We set the master mode to `local` (single-thread execution) and locked communication to standard, naturally unblocked developer ports (`7077` and `6060`).

---

### 🏆 6. The Final Permanent Solution (`sitecustomize.py`)
* **The Problem:** The configuration code block worked perfectly but was too verbose to paste at the top of every single script or notebook. Standard `spark-defaults.conf` files are ignored by `pip` installations.
* **The Cause:** `pip`-installed PySpark skips standard system files on boot, but the Python interpreter itself has a hidden startup hook.
* **Diagnostic Command (Run in Notebook Cell):**
  ```python
  import site
  print(site.getsitepackages())
  ```
* **The Resolution:** 
  1. Navigated to the printed directory path and opened `Lib/site-packages/`.
  2. Created a script named **`sitecustomize.py`** to serve as an automatic runtime interceptor.
  3. Placed our exact port limits, loopback configs, and Java 17 parameters inside it using `os.environ["PYSPARK_SUBMIT_ARGS"]`.

Now, your notebooks can remain completely clean and production-ready:
```python
from pyspark.sql import SparkSession

# Python automatically pre-injects your fixes on backend boot!
spark = SparkSession.builder.getOrCreate()

df = spark.createDataFrame([(1, 'a'), (2, 'b')], schema=['id', 'name'])
df.show()
```
